# [Anthropic API](https://platform.claude.com/docs/en/home)

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"
max_tokens = 100

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{"role": "user", "content": "Hello Claude, how are you?"}],
)

# Display the main content
print(f"Response: {message.content[0].text}")
print(f"Model: {message.model}")
print(f"Stop Reason: {message.stop_reason}")
print(f"Role: {message.role}")
print("Usage:")
print(f"  Input tokens: {message.usage.input_tokens}")
print(f"  Output tokens: {message.usage.output_tokens}")

Response: Hello! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?

Model: claude-haiku-4-5-20251001
Stop Reason: end_turn
Role: assistant

Usage:
  Input tokens: 14
  Output tokens: 32


## [In-Memory Math-tutor Chat Agent](https://platform.claude.com/docs/en/build-with-claude/working-with-messages)

### Selection Probability

Before generating each token (word), the model calculates a probability for every possible next word based on the context:

```
Input: "What do you think"
Next token probabilities:
  about  → 30%
  would  → 20%
  of     → 10%
  is     → 10%
  when   → 10%
  makes  → 5%
  we     → 5%
```

The model then **selects one token** based on these probabilities. A higher probability = more likely to be chosen.

### Temperature: Controlling Randomness

**Temperature** is a parameter that adjusts these selection probabilities. Using our example above:

| Range | Level | Effect on Example | Use Cases |
|---|---|---|---|
| **0.0 - 0.3** | **Low** | `about → 100%`, others → 0% | Factual responses, coding, data extraction, content moderation |
| **0.4 - 0.7** | **Medium** | `about → 30%`, `would → 20%`, `of → 10%` (unchanged) | Summarization, educational content, problem-solving, creative writing with constraints |
| **0.8 - 1.0+** | **High** | `about → 23%`, `would → 18%`, `of → 15%` (flattened distribution) | Brainstorming, creative writing, marketing, joke generation |

**Key principle:** Use lower temperatures when accuracy and consistency matter; use higher temperatures when diversity and creativity matter.

**In this tutor:** `temperature=1.0` keeps responses natural and varied without becoming nonsensical.

In [ ]:
def creative_tutor_math(messages: list[dict[str, str]]) -> str:
    """Generate a tutoring response using the math tutor system prompt.

    Args:
        messages: List of message dictionaries with 'role' and 'content' keys.

    Returns:
        str: The assistant's tutoring response.
    """
    system_prompt = """
        You are a patient math tutor.
        Do not directly answer a student's questions.
        Guide them to a solution step by step.
    """
    temperature = 0.0  # Adjust creativity level

    message = client.messages.create(
        model=model,
        max_tokens=200,
        messages=messages,
        system=system_prompt,
        temperature=temperature,
    )
    return message.content[0].text


messages: list[dict[str, str]] = []

# Use a 'while True' loop to run the chatbot forever, press ESC to stop
while True:
    try:
        # Get user input
        user_input = input("> ")
        print(">", user_input)

        # Add user input to the list of messages
        messages.append({"role": "user", "content": user_input})

        # Call Claude with the 'creative_tutor_math' function
        response = creative_tutor_math(messages)
        display(Markdown(response))

        # Add generated text to the list of messages
        messages.append({"role": "assistant", "content": response})
    except Exception as e:
        if "400" in str(e):
            break
        raise

def creative_tutor_math(messages: list[dict[str, str]]) -> str:
    """Generate a tutoring response using the math tutor system prompt.

    Args:
        messages: List of message dictionaries with 'role' and 'content' keys.

    Returns:
        str: The assistant's tutoring response.
    """
    system_prompt = """
        You are a patient math tutor.
        Do not directly answer a student's questions.
        Guide them to a solution step by step.
    """
    temperature = 0.0  # Adjust creativity level

    message = client.messages.create(
        model=model,
        max_tokens=200,
        messages=messages,
        system=system_prompt,
        temperature=temperature,
    )
    return message.content[0].text


messages: list[dict[str, str]] = []

# Use a 'while True' loop to run the chatbot forever, press ESC to stop
while True:
    try:
        # Get user input
        user_input = input("> ")
        print(">", user_input)

        # Add user input to the list of messages
        messages.append({"role": "user", "content": user_input})
        
        # Call Claude with the 'creative_tutor_math' function
        response = creative_tutor_math(messages)
        display(Markdown(response))
        
        # Add generated text to the list of messages
        messages.append({"role": "assistant", "content": response})
    except Exception as e:
        if "400" in str(e):
            break
        raise

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Hello Claude, how are you? What is the weather like today?"}
    ],
    temperature=1.0,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

## Structured data

In [ ]:
import json

# Structured data

response = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"content": "Generate a random user data in json format", "role": "user"},
        {
            "content": "Here is it, just one user, with fields: user_id, name, email, and phone. Nothing more...\n```json",
            "role": "assistant",
        },
    ],
    stop_sequences=["```"],
)

print("AI message:", response.content[0].text)

event_bridge_rule = json.loads(response.content[0].text)
print("Json parsed:", event_bridge_rule)

AI message: 
{
  "user_id": "USR-7284-9150",
  "name": "Sarah Mitchell",
  "email": "sarah.mitchell@email.com",
  "phone": "+1-555-0147"
}

Json parsed: {'user_id': 'USR-7284-9150', 'name': 'Sarah Mitchell', 'email': 'sarah.mitchell@email.com', 'phone': '+1-555-0147'}


## Prompt evaluation